例:
```bash
f=../data/sample_151507_anno.h5
for i in {1..5}
    do
        python -u run_DSSC.py --n_clusters -1 --data_file $f --save_dir out_151507 \
            --final_labels pred.csv --final_latent_file latent.csv --run $i \
            --ml_file ../example_constraints/sample_151507_mlFromMarks_test.txt \
            --cl_file ../example_constraints/sample_151507_clFromMarks_test.txt
    done
```

# 节点 1：数据加载与处理

src/run_DSSC.py:147，model.train_model(...) 被调用的那一刻。三个输入文件都已完成加载/变换，正要第一次进入模型。

## 1. f=../data/sample_151507_anno.h5

### 读取模块：run_DSSC.py:53-62

| h5 字段 | 原始形状/类型	   | 说明                     |
| ------ | ---------------- | ------------------------ | 
| X	| (4226, 24402) float64 | 原始 counts 矩阵 | 
| Pos | (2, 4226) int32 | 空间坐标（2 行） | 
| Y | (4226,) bytes | 金标准层标签 Layer1..6/WM/NA | 
| Gene | (24402,) | 注意：当前代码没有读取 Gene（只读 X/Pos/Y） | 

In [20]:
import h5py
import numpy as np

data_mat = h5py.File("../../data/sample_151507_anno.h5")
x = np.array(data_mat['X'])
pos = np.array(data_mat['Pos'])
if pos.shape[1] != 2: 
    pos = pos.T
y = np.array(data_mat['Y'])
data_mat.close()

print(x.shape)
print(pos.shape)
print(y.shape)

(4226, 24402)
(4226, 2)
(4226,)


/tmp/ipykernel_591778/4125188590.py:4: H5pyDeprecationWarning: The default file mode will change to 'r' (read-only) in h5py 3.0. To suppress this warning, pass the mode you need to h5py.File(), or set the global default h5.get_config().default_file_mode, or set the environment variable H5PY_DEFAULT_READONLY=1. Available modes are: 'r', 'r+', 'w', 'w-'/'x', 'a'. See the docs for details.
  data_mat = h5py.File("../../data/sample_151507_anno.h5")


In [12]:
print(x, '\n')
print(pos, '\n')
print(y)

[[ 0.  0.  0. ...  0.  5.  0.]
 [ 0.  0.  0. ...  0. 50.  0.]
 [ 0.  0.  0. ...  0. 22.  0.]
 ...
 [ 0.  0.  0. ...  0.  7.  0.]
 [ 0.  0.  0. ...  1. 26.  0.]
 [ 0.  0.  0. ...  0. 31.  0.]] 

[[  0  16]
 [ 50 102]
 [  3  43]
 ...
 [ 60  30]
 [ 45  27]
 [  7  51]] 

[b'Layer1' b'Layer3' b'Layer1' ... b'WM' b'Layer6' b'Layer1']


### 变换过程：
1. run_DSSC.py:56-57：因为 Pos.shape[1] != 2，执行 pos = pos.T → (4226, 2)。（这正是 README 里对 osmFISH 修复的同一处逻辑。）
2. run_DSSC.py:65-69：f = where(Y != "NA")，剔除 5 个 NA 细胞 → x=(4221,24402)、pos=(4221,2)、y=(4221,)。注意：这里重排了细胞索引，而 ml/cl 文件也是按同一顺序、在剔 NA 后生成的，所以索引仍能对齐。
3. run_DSSC.py:72-75：n_clusters = len(unique(y)) = 7（Layer1–6 + WM）。Y 到此只用于确定簇数。
4. 构造 AnnData（run_DSSC.py:87-92）：obs 写入 cell_labels、batch=1、pos_x/pos_y。
5. preprocessing_rna（preprocess.py:78-151）：
- 丢弃 ERCC/MT-/mt- 基因（preprocess.py:118）；
- count_X = adata.X（保留原始 counts 副本，preprocess.py:124）；
- normalize_per_cell（preprocess.py:133）→ log1p（preprocess.py:137）→ 选 2000 个高变基因并 subset（preprocess.py:141）；
- 返回 adata=(4221,2000)（log 归一化 HVG）和 count_X=(4221,2000)（原始 counts，仅 HVG 列）。
- size_factors = n_counts（归一化前的每细胞总 counts，preprocess.py:134）。
6. run_DSSC.py:94-99：pos_=(4221,2) 从 obs 取回；y 重编码为 cat.codes 0–6。
进入模型的分流（都在 model.train_model，DSSC.py:148）：

In [21]:
import pandas as pd
import scanpy as sc

print(f'需剔除细胞数: {(y.astype(np.str) == "NA").sum()}') # 需剔除的细胞数：5
# 2 剔除 NA
f = np.where(y.astype(np.str) != "NA")[0]
y = y[f]
x = x[f,:]
pos = pos[f,:]
pos = pos.astype(np.float)
# 3 获取簇数
n_clusters = np.shape(np.unique(y))[0] # 共有几簇？
print(f'簇数: {n_clusters}')
# 4 构建 Anndata 对象
obs = pd.DataFrame()
obs['cell_labels'] = pd.Categorical(y, categories=np.unique(y), ordered=False)
obs['batch'] = pd.Categorical(np.repeat(1, x.shape[0]), categories=[1], ordered=False)
obs['pos_x'] = pos[:,0]
obs['pos_y'] = pos[:,1]
adata = sc.AnnData(X=x, obs=obs)

/home/liuzy/Software/miniconda3/envs/dssc-4090/lib/python3.8/site-packages/setuptools_scm/_integration/setuptools.py:31: RuntimeWarning: 
ERROR: setuptools==59.8.0 is used in combination with setuptools_scm>=8.x

Your build configuration is incomplete and previously worked by accident!
setuptools_scm requires setuptools>=61

Suggested workaround if applicable:
 - migrating from the deprecated setup_requires mechanism to pep517/518
   and using a pyproject.toml to declare build dependencies
   which are reliably pre-installed before running the build tools

  warnings.warn(


需剔除细胞数: 5
簇数: 7


/home/liuzy/Software/miniconda3/envs/dssc-4090/lib/python3.8/site-packages/anndata/_core/anndata.py:120: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [ ]:
adata

AnnData object with n_obs × n_vars = 4221 × 24402
    obs: 'cell_labels', 'batch', 'pos_x', 'pos_y'

In [25]:
import sys, pathlib
sys.path.append(str(pathlib.Path("../../src").resolve()))
from preprocess import preprocessing_rna

# 5 预处理，此处调用了 preprocess.py 中的 preprocessing_rna 函数
adata, count_X = preprocessing_rna(adata=adata, n_top_features=2000) 
# adata 经过了去除 mt，normalize，log1p，取 HVG，subset 等操作
# count_X 是原始 counts 副本

# 6
pos_ = np.zeros([adata.n_obs, 2])
pos_[:,0] = adata.obs["pos_x"]
pos_[:,1] = adata.obs["pos_y"]
y = np.array(adata.obs["cell_labels"].cat.codes)
print("Size of after filtering", adata.X.shape)
print("Size of after filtering", count_X.shape)

Trying to set attribute `.obs` of view, copying.


Size of after filtering (4221, 2000)
Size of after filtering (4221, 2000)


/home/liuzy/Software/miniconda3/envs/dssc-4090/lib/python3.8/site-packages/scanpy/preprocessing/_highly_variable_genes.py:504: FutureWarning: Slicing a positional slice with .loc is not supported, and will raise TypeError in a future version.  Use .loc with labels or .iloc with positions instead.
  df.loc[:n_top_genes, 'highly_variable'] = True


### 进入模型的分流（都在 model.train_model，DSSC.py:148）：
- adata.X（4221×2000，log 归一化）→ 编码器输入 + 图节点特征（G.ndata['feat']，DSSC.py:151）。
- count_X（原始 counts）→ ZINB 重建目标（X_raw_v，DSSC.py:152/158）。
- size_factors → ZINB 的 scale_factor（DSSC.py:153/159）。

## 2. sample_151507_mlFromMarks_test.txt / sample_151507_clFromMarks_test.txt

### 读取模块：run_DSSC.py:120-136
- np.loadtxt 读入：ml 形状 (26203, 2)，cl 形状 (31828, 2)，每行是细胞索引对。
- run_DSSC.py:126-128 / 134-136：ml_ind1 = col0 - 1、ml_ind2 = col1 - 1（R 从 1 计数，故减 1），cl 同理。
- 这两个数组此时只是索引池，还没进模型。

In [29]:
anchors_ml = np.loadtxt("../../example_constraints/sample_151507_mlFromMarks_test.txt")
anchors_ml = anchors_ml.astype(np.int)
ml_ind1 = anchors_ml[:,0] - 1
ml_ind2 = anchors_ml[:,1] - 1 

anchors_cl = np.loadtxt("../../example_constraints/sample_151507_clFromMarks_test.txt")
anchors_cl = anchors_cl.astype(np.int)
#index -1 since here we build constraints from R
cl_ind1 = anchors_cl[:,0] - 1
cl_ind2 = anchors_cl[:,1] - 1

### 进入模型的分流：传入 model.fit（run_DSSC.py:151-154 → DSSC.py:204-206），一路传到 pairwise_loss（DSSC.py:190-196）：
- ML → qbatch[ml_ind1]、qbatch[ml_ind2]（DSSC.py:313-315）→ -log(Σ p1·p2)；
- CL → qbatch[cl_ind1]、qbatch[cl_ind2]（DSSC.py:328-330）→ -log(1-Σ p1·p2)。

## 图结构（由 Pos 派生，非文件输入）

run_DSSC.py:102-107：A = kneighbors_graph(pos_, 20, include_self=True)，A_n = D^{-1/2} A D^{-1/2}。A_n 变成 dgl 图，喂给 GAT 编码器；A 在预训练阶段实际未用。A0、p_（run_DSSC.py:110-118）只用于评估，不参与训练

In [58]:
from sklearn.neighbors import kneighbors_graph
from scipy import sparse
from preprocess import norm_adj
from scipy import spatial

print(pos_)
A = kneighbors_graph(pos_, 20, mode="connectivity", metric="euclidean", include_self=True, n_jobs=-1)
A = A.toarray()    
A = sparse.csr_matrix(A)
A_n = norm_adj(A)

###Get kneighbors_graph from spatial dist (this is just for evaluation)
k0 = 20 #we now use a k=20 graph to evaluate
A0 = kneighbors_graph(pos_, k0, mode="connectivity", metric="euclidean", include_self=False, n_jobs=-1)
A0 = A0.toarray()

[[  0.  16.]
 [ 50. 102.]
 [  3.  43.]
 ...
 [ 60.  30.]
 [ 45.  27.]
 [  7.  51.]]


In [ ]:
dist = spatial.distance_matrix(pos_,pos_)
p_ = []
for i in range(dist.shape[0]):
    idx = np.argpartition(dist[i], k0)[:k0]
    p_.append(idx)

In [59]:
print(f"A shape: {A.shape}")

print(A[:5, :5].toarray(), '\n')
print(A_n[:5, :5].toarray(), '\n')
print(A0[:5, :5], '\n')

print(dist[:5, :5])

A shape: (4221, 4221)
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 1.]] 

[[0.05 0.   0.   0.   0.  ]
 [0.   0.05 0.   0.   0.  ]
 [0.   0.   0.05 0.   0.  ]
 [0.   0.   0.   0.05 0.  ]
 [0.   0.   0.   0.   0.05]] 

[[0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]] 

[[ 0.         99.47864092 27.16615541 59.07622195 43.56604182]
 [99.47864092  0.         75.43208866 83.48652586 93.26306879]
 [27.16615541 75.43208866  0.         60.92618485 52.49761899]
 [59.07622195 83.48652586 60.92618485  0.         18.86796226]
 [43.56604182 93.26306879 52.49761899 18.86796226  0.        ]]


# 给 train_model 时的模型输入总览

run_DSSC.py:138-140 构造模型：input_dim = adata.n_vars = 2000，encodeLayer=[128]，decodeLayer=[128]，encodeHead=3，z_dim=32。编码器结构为：GATConv(2000→128, 3 heads) → 多头求和 → BatchNorm1d(128) →（再 BatchNorm）→ GATConv(128→32, 3 heads) → 多头求和，得到每个细胞的 32 维隐表示 z。
两个我在断点处发现的“可疑点”（先记下，后面会用到）：
1. DSSC.py:93 构造编码器时没有把 encodeConcat 传进去，GATEncoder 始终用默认 concat=False，所以 --concat True 实际不生效（与 .logs/10021641.log 打印 BatchNorm1d(128) 而非 384 一致）。
2. DSSC.py:74-75 的 else 分支把 layers[3*i+1]（BatchNorm）连续调用了两次，而 layers[3*i+2]（ELU）从未被调用。

In [60]:
# 导出对象
import scipy.sparse as sp, pathlib

p = pathlib.Path("data")
p.mkdir(exist_ok=True)

adata.write(p/"adata.h5ad")

sp.save_npz(p/"A.npz", A)
sp.save_npz(p/"A_n.npz", A_n)

np.savez(p/"arrays.npz", count_X=count_X, pos_=pos_, y=y,
         ml_ind1=ml_ind1, ml_ind2=ml_ind2, cl_ind1=cl_ind1, cl_ind2=cl_ind2,
         n_clusters=n_clusters, A0=sp.csr_matrix(A0), p_=np.array(p_, dtype=object))